# 04_01 Layer normalization

*📖 Book: section 4.2*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
>
> **This notebook:** Numbers passed through many layers drift too small or too big. Layer norm resets each row so a deep model stays steady.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why the numbers flowing through a deep model can drift too big or too small
- How layer norm fixes each row: mean 0 and variance 1, worked by hand on 4 numbers
- How `nn.LayerNorm` does the same and learns a scale and a shift
- That our small `model.py` has no layer norm, and why it gets away with it

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **layer norm** | short for layer normalization: a step that rescales each row of numbers to a fixed mean and spread, then lets the model learn a scale and a shift | `nn.LayerNorm(32)` |
| **mean** | the average: add the numbers, divide by how many | the mean of 3, 5, 7, 9 is 6 |
| **variance** | how spread out the numbers are: the average of each number's squared distance from the mean | for 3, 5, 7, 9 it's 5 |
| **standard deviation** | the square root of the variance: a typical distance from the mean, in the same units as the numbers | √5 ≈ 2.24 |
| **scale and shift** | two learned numbers per position in the row: multiply by the scale, then add the shift | `scale = 1, shift = 0` at the start |

## 3. The idea

### ❓ What goes wrong without it?

Each layer multiplies the numbers by its weights. Do that again and again, and the numbers can shrink towards 0 or grow huge. If they shrink, there's nothing left to learn from. If they grow, learning gets jumpy (like the learning-rate spike in [05_02](../05_training_on_the_odyssey/05_02_learning_rate.ipynb)).

We can watch it happen. Take the real five rows of `ἄνδρα`, pass them through 8 plain layers (random numbers, no norm), and after each layer measure the spread (the **standard deviation**) of the numbers in a row.

👉 **Predict first:** the 32 numbers of each row start with a spread of about 1. After 8 layers, will it be about 1, much smaller, or much bigger?

In [ ]:
import torch
from torch import nn

from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
print("vocabulary size:", tokenizer.vocab_size)

**A new dimension: the batch.** The next cell makes `ids` with shape `(1, 5)` and `x` with shape `(1, 5, 32)`. That extra first number is the batch: a group of texts that the model processes at the same time (real training feeds in many at once, because the hardware is faster that way). We have just one text, so the batch size is 1. Read the shape as `(texts, letters, numbers per letter)`. Our `model.py` takes one text with no batch dimension (shape `(5,)`). The full model of this chapter takes a batch.

In [ ]:
torch.manual_seed(0)
embedding = nn.Embedding(tokenizer.vocab_size, 32)  # a new, untrained table
ids = torch.tensor([tokenizer.encode("ἄνδρα")])  # shape (1, 5): 1 text, 5 letters
x = embedding(ids)  # shape (1, 5, 32): 5 letters, 32 numbers each
print("ids:", tuple(ids.shape), " x:", tuple(x.shape))

In [ ]:
def spread(h):
    # the standard deviation of each row (each letter), averaged over the letters
    return h.std(dim=-1, unbiased=False).mean().item()


def run(scale, with_norm):
    torch.manual_seed(1)
    h, history = x.detach(), [spread(x)]
    for _ in range(8):
        layer = nn.Linear(32, 32, bias=False)
        with torch.no_grad():
            layer.weight *= scale  # scale 1 = PyTorch's normal start
        h = layer(h)
        if with_norm:
            h = nn.LayerNorm(32)(h)
        history.append(spread(h))
    return history, h


plain, drifted = run(1, with_norm=False)
big, _ = run(3, with_norm=False)
normed, _ = run(1, with_norm=True)
print("plain layers      :", f"1.00 → {plain[-1]:.3f}")
print("plain, weights ×3 :", f"1.00 → {big[-1]:.1f}")
print("with a norm each  :", f"1.00 → {normed[-1]:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

With plain layers the spread shrinks from about 1 to about 0.01. Make the weights three times bigger and it grows to about 65 instead. Put a norm after every layer and it stays at 1. And a real model has dozens of these steps, so the effect is even stronger. (`unbiased=False` means: divide by the number of values, as the layer norm does, not by one less.)

</details>

### ❓ Can I see it?

Same numbers, as lines. The vertical axis is a **log** scale (each gridline is 10 times the one below), because the lines are so far apart.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, BAD, GOOD, INK, studio_axes, studio_figure

fig = studio_figure("Layer norm", "The spread of a row, layer after layer", (9, 6))
ax = studio_axes(fig)
steps = range(9)
style = {"linewidth": 3, "marker": "o"}
ax.plot(steps, big, color=BAD, label="plain, weights ×3: grows", **style)
ax.plot(steps, plain, color=ACCENT, label="plain: shrinks", **style)
ax.plot(steps, normed, color=GOOD, label="with layer norm: stays at 1", **style)
ax.set_yscale("log")
ax.set_xlabel("number of layers passed")
ax.set_ylabel("spread (standard deviation) of a row")
ax.legend(frameon=False, labelcolor=INK, loc="upper left")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The two plain lines run away from 1 in opposite directions, and the green line stays flat. That flat line is the whole point of layer norm.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what does layer norm do to one row, by hand?

Two moves, for each row on its own:

1. **Centre it:** subtract the row's mean, so the new mean is 0.
2. **Rescale it:** divide by the row's standard deviation, so the new variance is 1.

Let's pretend the row for `ἄ` had only 4 numbers: `3, 5, 7, 9`.

```
row            3     5     7     9      mean = (3+5+7+9)/4 = 6
- mean        -3    -1    +1    +3      
squared        9     1     1     9      variance = 20/4 = 5, std = √5 ≈ 2.24
÷ std       -1.34 -0.45 +0.45 +1.34      the normalised row
```

👉 **Predict first:** what's the mean of the last row? And its variance?

In [ ]:
row = torch.tensor([[3.0, 5.0, 7.0, 9.0]])

mean = row.mean(dim=-1, keepdim=True)
variance = row.var(dim=-1, keepdim=True, unbiased=False)
eps = 1e-5  # a tiny number so we never divide by 0
by_hand = (row - mean) / torch.sqrt(variance + eps)

print("mean:", mean.item(), " variance:", variance.item())
print("by hand:", [round(v, 2) for v in by_hand[0].tolist()])
new_mean = by_hand.mean().item()
new_variance = by_hand.var(unbiased=False).item()
print("new mean:", round(new_mean, 6), " new variance:", round(new_variance, 4))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Mean 6 and variance 5, as in the picture. The new row is `-1.34, -0.45, 0.45, 1.34`, with mean 0 and variance 1 (a tiny bit under, because of `eps`). `dim=-1` means "along the last direction", so the numbers of one row. And `keepdim=True` keeps the answer shaped as a column, so we can subtract it from the row.

</details>

### ❓ Step 2: what does `nn.LayerNorm` add?

It does the same two moves, then multiplies by a learned **scale** and adds a learned shift, one pair for each number in the row. They start at scale 1 and shift 0, so at first the result is just our by-hand answer. Training can change them later, if the model finds that a different centre or spread suits that place better.

👉 **Predict first:** will `nn.LayerNorm(4)` give the same row as our by-hand answer?

In [ ]:
norm = nn.LayerNorm(4)
out = norm(row)
print("same as by hand?", torch.allclose(out, by_hand, atol=1e-6))
print("scale:", norm.weight.tolist())
print("shift:", norm.bias.tolist())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True`, a scale of four ones and a shift of four zeros. (`weight` and `bias` are PyTorch's names for the scale and the shift.)

</details>

### ❓ Your turn: can you normalise another row by hand?

Here's a row with much bigger numbers: `[10, 20, 30, 40]`. Do to it what step 1 did to `[3, 5, 7, 9]`.

👉 **Predict first:** these numbers are about four times as big. Will the normalised row be bigger too?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
other_row = torch.tensor([[10.0, 20.0, 30.0, 40.0]])

# 👉 do: subtract the row's mean, then divide by the square root of (variance + eps)
# 💡 hint: mean and var both take dim=-1, keepdim=True; var also takes unbiased=False
other_normed = None

In [ ]:
from homer_ai.lessons import check

check(
    "the normalised row",
    other_normed,
    norm(other_row),
    coach="Numbers near -1.16 and 1.16 mean var() divided by n - 1: pass "
    "unbiased=False. Numbers like -0.12 mean you divided by the variance itself: "
    "take its square root first.",
)

<details><summary>💡 Hint</summary>

Step 1 has all three lines. Write `other_row` where it has `row`, and keep `eps`.

</details>

<details><summary>✅ Answer</summary>

```python
other_row = torch.tensor([[10.0, 20.0, 30.0, 40.0]])

other_mean = other_row.mean(dim=-1, keepdim=True)
other_variance = other_row.var(dim=-1, keepdim=True, unbiased=False)
other_normed = (other_row - other_mean) / torch.sqrt(other_variance + eps)
```

`[-1.34, -0.45, 0.45, 1.34]`, exactly what `[3, 5, 7, 9]` gave. Both rows are four evenly spaced numbers, and once the mean and the spread are taken out, that pattern is all that's left. This is why layer norm keeps a deep model steady: however big or small a row has drifted, it comes out at the same size.

</details>

### ❓ Step 3: what does it do to each letter of `ἄνδρα`?

Each letter's row is normalised on its own. Letters never mix here. So let's take the shrunken rows from our 8-layer test (`drifted`) and normalise them.

```
before:  ἄ → 32 tiny numbers      after:  ἄ → 32 numbers, mean 0, spread 1
         ν → 32 tiny numbers              ν → 32 numbers, mean 0, spread 1   …
```

👉 **Predict first:** the spread of each row, before and after.

In [ ]:
after = nn.LayerNorm(32)(drifted)

print("letter   spread before   mean after   spread after")
for i, letter in enumerate("ἄνδρα"):
    before_spread = drifted[0, i].std(unbiased=False).item()
    mean_after = after[0, i].mean().item()
    spread_after = after[0, i].std(unbiased=False).item()
    row = f"  {letter}        {before_spread:.4f}"
    print(row, f"       {mean_after:+.4f}        {spread_after:.4f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Every row had a spread of about 0.01. After the norm each one has a spread of 0.88 to 0.97, with a mean of 0. Why not exactly 1? Because the tiny `eps` we add starts to matter when the spread is this small (about 0.01, so a variance of about 0.0001, close to `eps`). Each letter's row keeps its own pattern. Only its centre and spread are reset.

</details>

### ❓ What does layer norm cost?

Two numbers (scale and shift) per number in the row. At our size that's 2 × 32 = 64 weights for each layer norm. A block has two of them and the model has one more at the end, so it's cheap.

👉 **Predict first:** how many weights is `nn.LayerNorm(32)`?

In [ ]:
big_norm = nn.LayerNorm(32)
print("weights:", sum(p.numel() for p in big_norm.parameters()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

64: that's 32 scales and 32 shifts.

</details>

### ❓ Does our `model.py` have layer norm?

No. It has only one attention step, so the numbers don't travel through many layers and nothing drifts much. A tall stack does need it, which is why the block in [04_04](04_04_one_block.ipynb) has two. We don't change `model.py` here.

## 5. Recap

- Passing numbers through many layers makes a row's spread shrink or grow (we measured 1 → 0.01, and 1 → 65).
- Layer norm gives each row mean 0 and variance 1, on its own, then applies a learned scale and shift (64 weights for a row of 32).
- Our `model.py` has no layer norm. The block in 04_04 has two.

**You can now:**

- normalise a row by hand: subtract the mean, divide by the spread
- use `nn.LayerNorm` and say what its scale and shift are for
- measure whether a row's spread shrinks or grows from layer to layer

**Next:** [04_02: Feed-forward](04_02_feed_forward.ipynb): what each letter does with what attention gave it.